### Импорты и пути

In [16]:
import sys
sys.path.append('../src')

import torch
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import importlib
import time

from tqdm.auto import tqdm

import rl_train as rl_train_mod
importlib.reload(rl_train_mod)
from rl_train import train_rl

import judge as judge_mod
importlib.reload(judge_mod)
from judge import JudgeModel

import reward_functions as rf_mod
importlib.reload(rf_mod)
from reward_functions import RMReward, RewardFunction

from dataset import build_item_vocab
from model import create_model
from reward_model import create_reward_model
from rm_index import RMIndex
from my_evaluate import GPT2Predictor, evaluate_gpt2_fast
from metrics import (
    build_eval_targets, build_context,
    recall_at_k, ndcg_at_k, average_precision_at_k,
)

PROC = Path('../data/processed')
RESULTS = Path('../results')
RESULTS_R2 = RESULTS / 'rl_r2_results'
RESULTS_R2.mkdir(exist_ok=True)
RESULTS_R3 = RESULTS / 'rl_r3_results'
RESULTS_R3.mkdir(exist_ok=True)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

Device: cuda


### Данные и словарь

In [2]:
train_df = pd.read_parquet(PROC / 'train.parquet')
val_df   = pd.read_parquet(PROC / 'val.parquet')
test_df  = pd.read_parquet(PROC / 'test.parquet')

item2idx, idx2item = build_item_vocab(train_df)
vocab_size = len(item2idx) + 2
print(f"Vocab size: {vocab_size}")

Vocab size: 14937


### π_ref и RM

In [3]:
def load_gpt2_policy(ckpt):
    m = create_model(vocab_size=vocab_size)
    m.load_state_dict(torch.load(ckpt, map_location='cpu'))
    return m

ref_policy = load_gpt2_policy(RESULTS / 'checkpoints_v2' / 'best_model.pt').to(device).eval()
for p in ref_policy.parameters():
    p.requires_grad = False

rm = create_reward_model(vocab_size=vocab_size,
                         init_from=RESULTS / 'checkpoints_v2' / 'best_model.pt')
rm.load_state_dict(torch.load(RESULTS / 'checkpoints_rm_v2' / 'best_model.pt',
                              map_location='cpu'))
rm = rm.to(device).eval()
for p in rm.parameters():
    p.requires_grad = False
print(f"RM: {sum(p.numel() for p in rm.parameters()):,} params, frozen")

[transformers] Model config: bos_token_id must be `None` or an integer within the vocabulary (between 0 and 14936), got 50256. This may result in unexpected behavior.
[transformers] Model config: eos_token_id must be `None` or an integer within the vocabulary (between 0 and 14936), got 50256. This may result in unexpected behavior.


[create_reward_model] Backbone: загружено 28 тензоров
[create_reward_model] candidate_emb инициализирован из wte
RM: 4,268,289 params, frozen


### RMIndex

In [4]:
all_df = pd.concat([train_df, val_df, test_df], ignore_index=True)
idx = RMIndex(all_df, item2idx, t1='2021-07-24', t2='2021-08-04')

RMIndex: 4,632,197 событий
  train: 2,789,962
  val:   371,021
  test:  474,130


### Helper-функции (score_candidates, _batched_policy_logits, fixed_eval)

In [5]:
@torch.no_grad()
def _batched_policy_logits(policy, contexts, item2idx, max_len=50, device='cuda',
                            batch_b=512, apply_masking=True, desc=None):
    B = len(contexts)
    out = []
    n_batches = (B + batch_b - 1) // batch_b
    pbar = tqdm(range(0, B, batch_b),
                desc=desc or "policy logits",
                total=n_batches, leave=False)
    for s in pbar:
        end = min(s + batch_b, B)
        inp = torch.zeros(end - s, max_len, dtype=torch.long, device=device)
        for i, ctx in enumerate(contexts[s:end]):
            recent = ctx[-max_len:]
            indexed = [item2idx.get(int(x), 1) for x in recent]
            if indexed:
                inp[i, -len(indexed):] = torch.tensor(indexed, device=device)
        try:
            logits = policy(inp, logits_to_keep=1).logits[:, -1, :].clone()
        except TypeError:
            logits = policy(inp).logits[:, -1, :].clone()
        if apply_masking:
            logits[:, 0] = -1e9
            logits[:, 1] = -1e9
            for i, ctx in enumerate(contexts[s:end]):
                s_items = [item2idx[int(x)] for x in ctx if int(x) in item2idx]
                if s_items:
                    logits[i, torch.tensor(s_items, device=device)] = -1e9
        out.append(logits)
    return torch.cat(out, dim=0)


@torch.no_grad()
def score_candidates(rm, contexts, cand_idx, item2idx,
                     max_len=50, chunk=8192, batch_b=256, device='cuda', desc=None):
    B = len(contexts)
    L = max_len
    M = cand_idx.shape[1]
    hist = torch.zeros(B, L - 1, dtype=torch.long, device=device)
    for i, ctx in enumerate(contexts):
        recent = ctx[-(L - 1):] if ctx else []
        indexed = [item2idx.get(int(x), 1) for x in recent]
        if indexed:
            hist[i, -len(indexed):] = torch.tensor(indexed, device=device)
    out = []
    n_batches = (B + batch_b - 1) // batch_b
    pbar = tqdm(range(0, B, batch_b),
                desc=desc or f"RM score {B}x{M}",
                total=n_batches, leave=False)
    for s in pbar:
        end = min(s + batch_b, B)
        h = hist[s:end].unsqueeze(1).expand(-1, M, -1)
        c = cand_idx[s:end].unsqueeze(2)
        seq = torch.cat([h, c], dim=2).reshape(-1, L)
        for s2 in range(0, seq.size(0), chunk):
            out.append(torch.sigmoid(rm(seq[s2:s2 + chunk])))
    return torch.cat(out).view(B, M)


@torch.no_grad()
def fixed_eval(policy, ref_policy, rm, user_ids, contexts, user_targets,
               item2idx, idx2item, K=10, max_len=50, device='cuda'):
    logits_pi  = _batched_policy_logits(policy,     contexts, item2idx, max_len, device)
    logits_ref = _batched_policy_logits(ref_policy, contexts, item2idx, max_len, device)
    logp_pi  = torch.log_softmax(logits_pi,  dim=-1)
    logp_ref = torch.log_softmax(logits_ref, dim=-1)
    kl = (logp_pi.exp() * (logp_pi - logp_ref)).sum(dim=-1).mean().item()
    topK = torch.topk(logits_pi, K, dim=-1).indices
    rm_scores = score_candidates(rm, contexts, topK, item2idx)
    mean_rm_top10 = rm_scores.mean().item()
    topK_lists = [[idx2item[int(i)] for i in row] for row in topK.cpu()]
    ndcg_v, recall_v, map_v = [], [], []
    for i, uid in enumerate(user_ids):
        targets = user_targets.get(uid, [])
        if not targets:
            continue
        preds = topK_lists[i]
        for t in targets:
            ndcg_v.append(ndcg_at_k(preds, t, K))
            recall_v.append(recall_at_k(preds, t, K))
            map_v.append(average_precision_at_k(preds, t, K))
    return dict(
        kl=kl, mean_rm_top10=mean_rm_top10,
        ndcg10=float(np.mean(ndcg_v)) if ndcg_v else 0.0,
        recall10=float(np.mean(recall_v)) if recall_v else 0.0,
        map10=float(np.mean(map_v)) if map_v else 0.0,
        n_targets=len(ndcg_v),
    )

### Контексты для fixed_eval (2000 test)

In [6]:
test_targets = build_eval_targets(test_df)
test_context = build_context(train_df, val_df)

test_users = test_targets['user_id'].unique()
np.random.seed(123)
eval_users = np.random.choice(test_users, size=2000, replace=False).tolist()

eval_contexts = []
for u in eval_users:
    ctx = test_context.get(u)
    items = ctx['items'] if ctx else []
    eval_contexts.append(list(items)[-50:])

user_targets = {}
for u in eval_users:
    user_targets[u] = test_targets[test_targets['user_id'] == u]['item_id'].tolist()

n_targets = sum(len(v) for v in user_targets.values())
print(f"Eval: {len(eval_users)} пользователей, {n_targets} таргетов")
assert all(len(c) > 0 for c in eval_contexts), "Пустой контекст - фильтр холодных должен был исключить"

Eval: 2000 пользователей, 2767 таргетов


### Base (π_ref) fixed_eval

In [7]:
print("[base] fixed_eval на π_ref ...")
base_fixed = fixed_eval(
    policy=ref_policy, ref_policy=ref_policy, rm=rm,
    user_ids=eval_users, contexts=eval_contexts, user_targets=user_targets,
    item2idx=item2idx, idx2item=idx2item,
)
print(f"[base] Er={base_fixed['mean_rm_top10']:.4f}  "
      f"NDCG@10={base_fixed['ndcg10']:.4f}  Recall@10={base_fixed['recall10']:.4f}")

[base] fixed_eval на π_ref ...


policy logits:   0%|          | 0/4 [00:00<?, ?it/s]

policy logits:   0%|          | 0/4 [00:00<?, ?it/s]

RM score 2000x10:   0%|          | 0/8 [00:00<?, ?it/s]

[base] Er=0.5096  NDCG@10=0.1361  Recall@10=0.2255


### Класс RefLogReward (R3)

In [9]:
class RefLogReward(RewardFunction):
    """
    R3: R0(h,a) + λ * log π_ref(a|h).
    
    Добавляет в награду релевантность через базовую политику. Поощряет
    айтемы, которые π_ref считает вероятными для данного пользователя,
    что ограничивает reward hacking на популярности.
    """
    def __init__(self, rm, ref_policy, item2idx, lambda_ref=0.1, max_len=50, device='cuda'):
        self.rm = rm
        self.ref_policy = ref_policy
        self.item2idx = item2idx
        self.lambda_ref = lambda_ref
        self.max_len = max_len
        self.device = device

    @torch.no_grad()
    def __call__(self, contexts, actions):
        B = len(contexts)

        # 1. R0 = sigmoid(RM(h + [a]))
        input_ids = torch.zeros(B, self.max_len, dtype=torch.long, device=self.device)
        for i, (ctx, act) in enumerate(zip(contexts, actions)):
            recent = ctx[-(self.max_len - 1):] if ctx else []
            indexed = [self.item2idx.get(int(x), 1) for x in recent]
            act_idx = self.item2idx.get(int(act), 1)
            seq = indexed + [act_idx]
            input_ids[i, -len(seq):] = torch.tensor(seq, device=self.device)
        r0 = torch.sigmoid(self.rm(input_ids))

        # 2. log π_ref(a | h)
        ctx_input = torch.zeros(B, self.max_len, dtype=torch.long, device=self.device)
        act_idxs = torch.zeros(B, dtype=torch.long, device=self.device)
        for i, (ctx, act) in enumerate(zip(contexts, actions)):
            recent = ctx[-self.max_len:] if ctx else []
            indexed = [self.item2idx.get(int(x), 1) for x in recent]
            if indexed:
                ctx_input[i, -len(indexed):] = torch.tensor(indexed, device=self.device)
            act_idxs[i] = self.item2idx.get(int(act), 1)

        logits_ref = self.ref_policy(ctx_input).logits[:, -1, :].clone()
        logits_ref[:, 0] = -1e9   # PAD
        logits_ref[:, 1] = -1e9   # UNK
        log_probs = torch.log_softmax(logits_ref, dim=-1)
        log_pi_ref = log_probs.gather(1, act_idxs.unsqueeze(1)).squeeze(1)
        log_pi_ref = torch.clamp(log_pi_ref, min=-50.0)

        # 3. R3 = R0 + λ * log π_ref
        return (r0 + self.lambda_ref * log_pi_ref).cpu()


rm_reward = RMReward(rm=rm, item2idx=item2idx, max_len=50, device=device)

# Проверка R3 = R0 + λ*log π_ref на 5 примерах
print("Проверка R3 = R0 + λ*log π_ref (λ=0.1):")
ref_log_reward_test = RefLogReward(rm, ref_policy, item2idx, lambda_ref=0.1,
                                    max_len=50, device=device)

sample_rows = np.random.choice(np.flatnonzero(idx.train_mask), size=5, replace=False)
ctxs, acts = [], []
for i in sample_rows:
    hi = idx.day_start[i]
    lo = max(idx.user_start[i], hi - 49)
    ctx = [idx2item[int(idx.items[j])] for j in range(lo, hi)]
    ctxs.append(ctx)
    acts.append(idx2item[int(idx.items[i])])

r0_vals = rm_reward(ctxs, acts).cpu()
r3_vals = ref_log_reward_test(ctxs, acts).cpu()

# Ручной расчёт log π_ref для сверки
ctx_input = torch.zeros(5, 50, dtype=torch.long, device=device)
act_idxs = torch.zeros(5, dtype=torch.long, device=device)
for i, (ctx, act) in enumerate(zip(ctxs, acts)):
    recent = ctx[-50:]
    indexed = [item2idx.get(int(x), 1) for x in recent]
    if indexed:
        ctx_input[i, -len(indexed):] = torch.tensor(indexed, device=device)
    act_idxs[i] = item2idx.get(int(act), 1)
logits_ref = ref_policy(ctx_input).logits[:, -1, :].clone()
logits_ref[:, 0] = -1e9; logits_ref[:, 1] = -1e9
log_pi = torch.log_softmax(logits_ref, dim=-1).gather(1, act_idxs.unsqueeze(1)).squeeze(1)
manual_r3 = r0_vals + 0.1 * log_pi.cpu()

for a, r0, r3, m in zip(acts, r0_vals, r3_vals, manual_r3):
    print(f"  item {a}: R0={r0:.4f}  log π_ref={log_pi[acts.index(a)]:.2f}  "
          f"R3={r3:+.4f}  manual={m:+.4f}  |diff|={abs((r3-m).item()):.2e}")

assert torch.allclose(r3_vals, manual_r3, atol=1e-5), "R3 != R0 + λ·log π_ref!"
print("RefLogReward корректен")

Проверка R3 = R0 + λ*log π_ref (λ=0.1):
  item 14703: R0=0.4982  log π_ref=-5.74  R3=-0.0754  manual=-0.0754  |diff|=0.00e+00
  item 10440: R0=0.3278  log π_ref=-3.35  R3=-0.0076  manual=-0.0076  |diff|=0.00e+00
  item 10837: R0=0.1853  log π_ref=-6.18  R3=-0.4325  manual=-0.4325  |diff|=0.00e+00
  item 5732: R0=0.4300  log π_ref=-7.34  R3=-0.3039  manual=-0.3039  |diff|=0.00e+00
  item 3076: R0=0.2060  log π_ref=-4.91  R3=-0.2849  manual=-0.2849  |diff|=0.00e+00
RefLogReward корректен


### RL R3

In [10]:
r3_configs = [
    (0.03, 2.0),   # слабое λ
    (0.1,  2.0),   # среднее λ — вероятный победитель
    (0.3,  2.0),   # сильное λ
    (0.1,  1.0),   # контроль: другое β
]
n_steps_r3 = 500
lr_r3 = 1e-5

r3_results = {}
histories_r3 = {}

for (lam, beta) in r3_configs:
    key = f"lam{lam}_beta{beta}"
    print(f"\n{'='*70}\nR3 λ={lam}, β={beta}\n{'='*70}")
    
    policy_r3 = load_gpt2_policy(RESULTS / 'checkpoints_v2' / 'best_model.pt').to(device)
    
    ref_log_reward = RefLogReward(rm, ref_policy, item2idx, lambda_ref=lam,
                                   max_len=50, device=device)
    
    t0 = time.time()
    history_r3 = train_rl(
        policy=policy_r3, ref_policy=ref_policy,
        reward_fn=ref_log_reward, rm_index=idx,
        idx2item=idx2item, item2idx=item2idx,
        n_steps=n_steps_r3, batch_size=64,
        lr=lr_r3, beta_kl=beta, G=8, top_m=200, max_len=50,
        device=device,
        save_dir=str(RESULTS / f'checkpoints_rl_r3_{key}'),
        seed=1, log_every=100, warmup_steps=50,
    )
    histories_r3[key] = history_r3
    print(f"Обучение: {time.time()-t0:.0f}s")
    
    metrics_r3 = fixed_eval(
        policy=policy_r3, ref_policy=ref_policy, rm=rm,
        user_ids=eval_users, contexts=eval_contexts, user_targets=user_targets,
        item2idx=item2idx, idx2item=idx2item,
    )
    r3_results[key] = {**metrics_r3, 'lambda': lam, 'beta': beta}
    print(f"fixed_eval R3 (λ={lam}, β={beta}): Er={metrics_r3['mean_rm_top10']:.4f}  "
          f"KL={metrics_r3['kl']:.4f}  NDCG@10={metrics_r3['ndcg10']:.4f}  "
          f"Recall@10={metrics_r3['recall10']:.4f}")
    
    del policy_r3
    torch.cuda.empty_cache()

print("\nR3 завершён")


R3 λ=0.03, β=2.0
Train contexts available: 2,789,962

Начинаем RL: 500 шагов, batch=64, lr=1e-05, β=2.0, G=8, M=200

Step  100/500 | pg_loss=-0.6721 | kl=0.0651 | mean_r=0.4096 | |A|=0.760 | max π_θ=0.144 | 31s
Step  200/500 | pg_loss=-0.7367 | kl=0.0819 | mean_r=0.4554 | |A|=0.765 | max π_θ=0.162 | 61s
Step  300/500 | pg_loss=-0.7761 | kl=0.0847 | mean_r=0.4445 | |A|=0.744 | max π_θ=0.168 | 91s
Step  400/500 | pg_loss=-0.7271 | kl=0.0838 | mean_r=0.4142 | |A|=0.763 | max π_θ=0.161 | 121s
Step  500/500 | pg_loss=-0.7915 | kl=0.0906 | mean_r=0.4152 | |A|=0.754 | max π_θ=0.152 | 152s

RL завершено: 500 шагов за 152s
Финальные метрики (последние 20 шагов):
  mean_reward = 0.4320
  kl          = 0.0852
Чекпоинты в: ..\results\checkpoints_rl_r3_lam0.03_beta2.0
Обучение: 152s


policy logits:   0%|          | 0/4 [00:00<?, ?it/s]

policy logits:   0%|          | 0/4 [00:00<?, ?it/s]

RM score 2000x10:   0%|          | 0/8 [00:00<?, ?it/s]

fixed_eval R3 (λ=0.03, β=2.0): Er=0.5476  KL=0.0979  NDCG@10=0.1332  Recall@10=0.2237

R3 λ=0.1, β=2.0
Train contexts available: 2,789,962

Начинаем RL: 500 шагов, batch=64, lr=1e-05, β=2.0, G=8, M=200

Step  100/500 | pg_loss=-0.9901 | kl=0.0712 | mean_r=0.0742 | |A|=0.782 | max π_θ=0.154 | 24s
Step  200/500 | pg_loss=-1.0216 | kl=0.0878 | mean_r=0.1372 | |A|=0.765 | max π_θ=0.174 | 47s
Step  300/500 | pg_loss=-1.0130 | kl=0.0889 | mean_r=0.1298 | |A|=0.770 | max π_θ=0.180 | 70s
Step  400/500 | pg_loss=-1.0622 | kl=0.0866 | mean_r=0.0960 | |A|=0.776 | max π_θ=0.173 | 93s
Step  500/500 | pg_loss=-1.0927 | kl=0.0970 | mean_r=0.0988 | |A|=0.751 | max π_θ=0.164 | 115s

RL завершено: 500 шагов за 115s
Финальные метрики (последние 20 шагов):
  mean_reward = 0.1133
  kl          = 0.0887
Чекпоинты в: ..\results\checkpoints_rl_r3_lam0.1_beta2.0
Обучение: 115s


...

fixed_eval R3 (λ=0.3, β=2.0): Er=0.5232  KL=0.1039  NDCG@10=0.1358  Recall@10=0.2273

R3 λ=0.1, β=1.0
Train contexts available: 2,789,962

Начинаем RL: 500 шагов, batch=64, lr=1e-05, β=1.0, G=8, M=200

Step  100/500 | pg_loss=-1.0510 | kl=0.0856 | mean_r=0.0744 | |A|=0.781 | max π_θ=0.166 | 23s
Step  200/500 | pg_loss=-1.0498 | kl=0.1456 | mean_r=0.1593 | |A|=0.756 | max π_θ=0.207 | 46s
Step  300/500 | pg_loss=-1.0403 | kl=0.1649 | mean_r=0.1715 | |A|=0.739 | max π_θ=0.218 | 69s
Step  400/500 | pg_loss=-1.1712 | kl=0.1888 | mean_r=0.1391 | |A|=0.775 | max π_θ=0.227 | 92s
Step  500/500 | pg_loss=-1.2190 | kl=0.1998 | mean_r=0.1315 | |A|=0.771 | max π_θ=0.214 | 114s

RL завершено: 500 шагов за 115s
Финальные метрики (последние 20 шагов):
  mean_reward = 0.1551
  kl          = 0.1866
Чекпоинты в: ..\results\checkpoints_rl_r3_lam0.1_beta1.0
Обучение: 115s


policy logits:   0%|          | 0/4 [00:00<?, ?it/s]

policy logits:   0%|          | 0/4 [00:00<?, ?it/s]

RM score 2000x10:   0%|          | 0/8 [00:00<?, ?it/s]

fixed_eval R3 (λ=0.1, β=1.0): Er=0.5385  KL=0.1953  NDCG@10=0.1342  Recall@10=0.2280

R3 завершён


### Лучшая (λ, β) для R3

In [11]:
valid_r3 = {k: m for k, m in r3_results.items()
            if m['ndcg10'] >= 0.97 * base_fixed['ndcg10']}

if valid_r3:
    best_key_r3 = max(valid_r3, key=lambda k: valid_r3[k]['mean_rm_top10'])
    print(f"Победитель R3: {best_key_r3}")
    print(f"λ={valid_r3[best_key_r3]['lambda']}, β={valid_r3[best_key_r3]['beta']}")
    print(f"Er={valid_r3[best_key_r3]['mean_rm_top10']:.4f}  "
          f"NDCG@10={valid_r3[best_key_r3]['ndcg10']:.4f}  "
          f"KL={valid_r3[best_key_r3]['kl']:.4f}")
else:
    best_key_r3 = min(r3_results, key=lambda k: abs(r3_results[k]['ndcg10'] - 0.97 * base_fixed['ndcg10']))
    print(f"Ни одна конфигурация не прошла порог NDCG ≥ 97% base.")
    print(f"Берём {best_key_r3} как минимальную потерю.")

print(f"\nВсе конфигурации R3:")
print(f"{'λ':>6}  {'β':>5}  {'Er':>7}  {'KL':>7}  {'NDCG@10':>8}  {'Recall@10':>10}")
print("-" * 60)
for k, m in r3_results.items():
    print(f"{m['lambda']:>6.2f}  {m['beta']:>5.1f}  {m['mean_rm_top10']:>7.4f}  "
          f"{m['kl']:>7.4f}  {m['ndcg10']:>8.4f}  {m['recall10']:>10.4f}")

Победитель R3: lam0.03_beta2.0
λ=0.03, β=2.0
Er=0.5476  NDCG@10=0.1332  KL=0.0979

Все конфигурации R3:
     λ      β       Er       KL   NDCG@10   Recall@10
------------------------------------------------------------
  0.03    2.0   0.5476   0.0979    0.1332      0.2237
  0.10    2.0   0.5381   0.1012    0.1348      0.2255
  0.30    2.0   0.5232   0.1039    0.1358      0.2273
  0.10    1.0   0.5385   0.1953    0.1342      0.2280


*Всё выглядит хорошо - Er в среднем примерно между 0.5 и 0.55 (R2 и R0) - примерно такие значения он, как компромисс, и должен иметь*

### Сравнение R3 с base на полном test

In [12]:
def compute_metrics(preds_dict, targets_df, k_values=(10, 50, 100)):
    max_k = max(k_values)
    accum = {f'{m}@{k}': [] for k in k_values for m in ['Recall', 'NDCG', 'MAP']}
    for _, row in targets_df.iterrows():
        preds = preds_dict.get(row['user_id'], [])[:max_k]
        t = row['item_id']
        for k in k_values:
            accum[f'Recall@{k}'].append(recall_at_k(preds, t, k))
            accum[f'NDCG@{k}'].append(ndcg_at_k(preds, t, k))
            accum[f'MAP@{k}'].append(average_precision_at_k(preds, t, k))
    return {k: float(np.mean(v)) for k, v in accum.items()}

@torch.no_grad()
def mean_rm_reward_on_full_test(preds_dict, context_dict, rm, item2idx,
                                 K=10, batch_size=512, device='cuda'):
    users = list(preds_dict.keys())
    ctxs = [context_dict.get(u, {}).get('items', []) for u in users]
    cand = torch.zeros(len(users), K, dtype=torch.long, device=device)
    for i, u in enumerate(users):
        preds = preds_dict[u][:K]
        for j, item in enumerate(preds):
            cand[i, j] = item2idx.get(int(item), 1)
    rewards = score_candidates(rm, ctxs, cand, item2idx,
                               batch_b=batch_size, device=device)
    return rewards.mean().item()

best_lam = r3_results[best_key_r3]['lambda']
best_beta_r3 = r3_results[best_key_r3]['beta']

print(f"[test] загружаем R3 λ={best_lam}, β={best_beta_r3}")
policy_best_r3 = load_gpt2_policy(
    RESULTS / f'checkpoints_rl_r3_{best_key_r3}' / 'final_model.pt'
).to(device).eval()

print("[test] R3 inference на полном test ...")
test_preds_r3 = evaluate_gpt2_fast(
    GPT2Predictor(policy_best_r3, item2idx, idx2item, max_len=50),
    test_targets, test_context, k=100, batch_size=512, exclude_seen=True,
)
r3_full = compute_metrics(test_preds_r3, test_targets)

print("[test] base inference на полном test ...")
test_preds_base = evaluate_gpt2_fast(
    GPT2Predictor(ref_policy, item2idx, idx2item, max_len=50),
    test_targets, test_context, k=100, batch_size=512, exclude_seen=True,
)
base_full = compute_metrics(test_preds_base, test_targets)

print(f"\n{'Метрика':<15} {'Base':>10} {'R3':>10} {'Δ':>10}")
print("=" * 50)
for m in base_full:
    d = (r3_full[m] / base_full[m] - 1) * 100 if base_full[m] > 0 else 0
    print(f"{m:<15} {base_full[m]:>10.4f} {r3_full[m]:>10.4f} {d:>+9.1f}%")

[test] загружаем R3 λ=0.03, β=2.0
[test] R3 inference на полном test ...
Уникальных юзеров для оценки: 132,323


predict_batch:   0%|          | 0/259 [00:00<?, ?it/s]

[test] base inference на полном test ...
Уникальных юзеров для оценки: 132,323


predict_batch:   0%|          | 0/259 [00:00<?, ?it/s]


Метрика               Base         R3          Δ
Recall@10           0.2192     0.2198      +0.3%
NDCG@10             0.1304     0.1301      -0.3%
MAP@10              0.1030     0.1024      -0.6%
Recall@50           0.3714     0.3706      -0.2%
NDCG@50             0.1638     0.1630      -0.5%
MAP@50              0.1101     0.1093      -0.7%
Recall@100          0.4592     0.4574      -0.4%
NDCG@100            0.1780     0.1771      -0.5%
MAP@100             0.1113     0.1105      -0.7%


### RM reward на полном test

In [13]:
print("[reward] RM reward топ-10 по базе ...")
reward_base_full = mean_rm_reward_on_full_test(test_preds_base, test_context, rm, item2idx, K=10)

print("[reward] RM reward топ-10 по R3 ...")
reward_r3_full = mean_rm_reward_on_full_test(test_preds_r3, test_context, rm, item2idx, K=10)

print(f"\n{'='*60}")
print(f"RM reward топ-10 (full test)")
print(f"{'='*60}")
print(f"  Base:  {reward_base_full:.4f}")
print(f"  R3:    {reward_r3_full:.4f}")
print(f"  Δ:     {(reward_r3_full/reward_base_full-1)*100:+.1f}%")

[reward] RM reward топ-10 по базе ...


RM score 132323x10:   0%|          | 0/259 [00:00<?, ?it/s]

[reward] RM reward топ-10 по R3 ...


RM score 132323x10:   0%|          | 0/259 [00:00<?, ?it/s]


RM reward топ-10 (full test)
  Base:  0.5092
  R3:    0.5444
  Δ:     +6.9%


### Валидация судьёй

In [14]:
judge = JudgeModel.load(RESULTS / 'checkpoints_judge' / 'judge')

print("[judge] оценка base ...")
judge_base_full  = judge.score_predictions_topk(test_preds_base, K=10)
mean_item_base   = judge.mean_item_mean_topk(test_preds_base, K=10)

print("[judge] оценка R3 ...")
judge_r3_full    = judge.score_predictions_topk(test_preds_r3, K=10)
mean_item_r3     = judge.mean_item_mean_topk(test_preds_r3, K=10)

d_judge = (judge_r3_full / judge_base_full - 1) * 100
d_itemmean = (mean_item_r3 / mean_item_base - 1) * 100

print(f"\n{'='*72}")
print(f"Сравнение оценок reward (full test)")
print(f"{'='*72}")
print(f"{'Метрика':<30} {'Base':>10} {'R3':>12} {'Δ':>10}")
print("-" * 72)
print(f"{'RM reward top-10':<30} {reward_base_full:>10.4f} {reward_r3_full:>12.4f} "
      f"{(reward_r3_full/reward_base_full-1)*100:>+9.1f}%")
print(f"{'Judge reward top-10':<30} {judge_base_full:>10.4f} {judge_r3_full:>12.4f} "
      f"{d_judge:>+9.1f}%")
print(f"{'Mean item_mean top-10':<30} {mean_item_base:>10.4f} {mean_item_r3:>12.4f} "
      f"{d_itemmean:>+9.1f}%")
print("-" * 72)

[judge] оценка base ...
[judge] оценка R3 ...

Сравнение оценок reward (full test)
Метрика                              Base           R3          Δ
------------------------------------------------------------------------
RM reward top-10                   0.5092       0.5444      +6.9%
Judge reward top-10                0.4817       0.5036      +4.5%
Mean item_mean top-10              0.4850       0.5189      +7.0%
------------------------------------------------------------------------


### Сохранение результатов R3

In [18]:
sweep_rows = [{'model': 'base', **base_fixed}]
for k, m in r3_results.items():
    sweep_rows.append({'model': k, **m})
pd.DataFrame(sweep_rows).to_csv(RESULTS_R3 / 'fixed_eval_r3_by_config.csv', index=False)

pd.DataFrame({
    'metric': ['reward_top10', 'NDCG@10', 'Recall@10', 'MAP@10'],
    'base': [reward_base_full, base_full['NDCG@10'], base_full['Recall@10'], base_full['MAP@10']],
    f'r3_{best_key_r3}': [reward_r3_full, r3_full['NDCG@10'], r3_full['Recall@10'], r3_full['MAP@10']],
}).to_csv(RESULTS_R3 / 'full_test_r3.csv', index=False)

pd.DataFrame({
    'metric': ['rm_reward_top10', 'judge_reward_top10', 'mean_item_mean_top10'],
    'base': [reward_base_full, judge_base_full, mean_item_base],
    f'r3_{best_key_r3}': [reward_r3_full, judge_r3_full, mean_item_r3],
    'delta_pct': [(reward_r3_full/reward_base_full-1)*100, d_judge, d_itemmean],
}).to_csv(RESULTS_R3 / 'judge_eval_r3.csv', index=False)

print(f"Сохранено в {RESULTS_R3}")

Сохранено в ..\results\rl_r3_results


### Сравнение R0 / R2 / R3

In [19]:
r0_full_path = RESULTS / 'full_test_reward_vs_relevance.csv'
r0_judge_path = RESULTS / 'judge_eval_full_test.csv'
r2_full_path = RESULTS_R2 / 'full_test_r2.csv'
r2_judge_path = RESULTS_R2 / 'judge_eval_r2.csv'

# Проверка, есть ли результаты R0 и R2
has_r0 = r0_full_path.exists() and r0_judge_path.exists()
has_r2 = r2_full_path.exists() and r2_judge_path.exists()

# Сбор данные
table = {
    'Base': {
        'RM reward': reward_base_full,
        'Judge reward': judge_base_full,
        'item_mean': mean_item_base,
        'NDCG@10': base_full['NDCG@10'],
        'Recall@10': base_full['Recall@10'],
        'MAP@10': base_full['MAP@10'],
    },
    f'R3 (λ={best_lam}, β={best_beta_r3})': {
        'RM reward': reward_r3_full,
        'Judge reward': judge_r3_full,
        'item_mean': mean_item_r3,
        'NDCG@10': r3_full['NDCG@10'],
        'Recall@10': r3_full['Recall@10'],
        'MAP@10': r3_full['MAP@10'],
    },
}

if has_r0:
    r0 = pd.read_csv(r0_full_path)
    r0j = pd.read_csv(r0_judge_path)
    table['R0 (β=2.0)'] = {
        'RM reward': float(r0.loc[r0['metric'] == 'reward_top10', 'rl_beta2.0'].values[0]),
        'Judge reward': float(r0j.loc[r0j['metric'] == 'judge_reward_top10', 'rl_beta2.0'].values[0]),
        'item_mean': float(r0j.loc[r0j['metric'] == 'mean_item_mean_top10', 'rl_beta2.0'].values[0]),
        'NDCG@10': float(r0.loc[r0['metric'] == 'NDCG@10', 'rl_beta2.0'].values[0]),
        'Recall@10': float(r0.loc[r0['metric'] == 'Recall@10', 'rl_beta2.0'].values[0]),
        'MAP@10': float(r0.loc[r0['metric'] == 'MAP@10', 'rl_beta2.0'].values[0]),
    }

if has_r2:
    r2 = pd.read_csv(r2_full_path)
    r2j = pd.read_csv(r2_judge_path)
    r2_col = [c for c in r2.columns if c.startswith('r2_')][0]
    r2j_col = [c for c in r2j.columns if c.startswith('r2_')][0]
    table['R2 (β=2.0)'] = {
        'RM reward': float(r2.loc[r2['metric'] == 'reward_top10', r2_col].values[0]),
        'Judge reward': float(r2j.loc[r2j['metric'] == 'judge_reward_top10', r2j_col].values[0]),
        'item_mean': float(r2j.loc[r2j['metric'] == 'mean_item_mean_top10', r2j_col].values[0]),
        'NDCG@10': float(r2.loc[r2['metric'] == 'NDCG@10', r2_col].values[0]),
        'Recall@10': float(r2.loc[r2['metric'] == 'Recall@10', r2_col].values[0]),
        'MAP@10': float(r2.loc[r2['metric'] == 'MAP@10', r2_col].values[0]),
    }

print(f"\n{'='*90}")
print(f"ФИНАЛЬНОЕ СРАВНЕНИЕ: Base / R0 / R2 / R3")
print(f"{'='*90}")
metrics = ['RM reward', 'Judge reward', 'item_mean', 'NDCG@10', 'Recall@10', 'MAP@10']
model_cols = list(table.keys())
header = f"{'Метрика':<16}" + "".join(f"{c:>20}" for c in model_cols)
print(header)
print("-" * 90)
for m in metrics:
    row = f"{m:<16}"
    base_val = table['Base'][m]
    for c in model_cols:
        val = table[c][m]
        if c == 'Base':
            row += f"{val:>20.4f}"
        else:
            d = (val / base_val - 1) * 100 if base_val != 0 else 0
            row += f"{val:>12.4f} ({d:>+5.1f}%)"
    print(row)
print("-" * 90)

# Сохранение
pd.DataFrame({
    'model': model_cols,
    'rm_reward': [table[c]['RM reward'] for c in model_cols],
    'judge_reward': [table[c]['Judge reward'] for c in model_cols],
    'item_mean': [table[c]['item_mean'] for c in model_cols],
    'ndcg10': [table[c]['NDCG@10'] for c in model_cols],
    'recall10': [table[c]['Recall@10'] for c in model_cols],
    'map10': [table[c]['MAP@10'] for c in model_cols],
}).to_csv(RESULTS_R3 / 'final_comparison.csv', index=False)
print(f"\nСохранено: {RESULTS_R3 / 'final_comparison.csv'}")


ФИНАЛЬНОЕ СРАВНЕНИЕ: Base / R0 / R2 / R3
Метрика                         Base  R3 (λ=0.03, β=2.0)          R0 (β=2.0)          R2 (β=2.0)
------------------------------------------------------------------------------------------
RM reward                     0.5092      0.5444 ( +6.9%)      0.5502 ( +8.0%)      0.5011 ( -1.6%)
Judge reward                  0.4817      0.5036 ( +4.5%)      0.5069 ( +5.2%)      0.4740 ( -1.6%)
item_mean                     0.4850      0.5189 ( +7.0%)      0.5245 ( +8.1%)      0.4731 ( -2.4%)
NDCG@10                       0.1304      0.1301 ( -0.3%)      0.1290 ( -1.1%)      0.1305 ( +0.0%)
Recall@10                     0.2192      0.2198 ( +0.3%)      0.2179 ( -0.6%)      0.2188 ( -0.2%)
MAP@10                        0.1030      0.1024 ( -0.6%)      0.1016 ( -1.4%)      0.1032 ( +0.2%)
------------------------------------------------------------------------------------------

Сохранено: ..\results\rl_r3_results\final_comparison.csv
